In [1]:
import numpy as np
from epic.core.compilation.qec_compiler import QECCompiler
from epic.core.data_structure.pauli import PauliEigenState
from epic.core.language.qec_gadget import AllocCode, FreeCode
from epic.modules.qec_gadgets.logical_resets.init_code import InitCode
from epic.modules.qec_gadgets.readout_code import ReadoutCode
from epic.modules.qec_gadgets.transversal_gates.transversal_h import TransversalH
from epic.modules.stabilizers_codes.css_code import CSSCode

# Steane [[7,1,3]] code  –  Hx = Hz = parity-check matrix of the [7,4,3] Hamming code
H = np.array(
    [[0, 0, 0, 1, 1, 1, 1],
     [0, 1, 1, 0, 0, 1, 1],
     [1, 0, 1, 0, 1, 0, 1]],
    dtype=np.uint8,
)
lX = lZ = [1, 1, 1, 0, 0, 0, 0]

# Two independent copies – different code_name so their Tanner-graph node tags
# remain disjoint when the cone code merges both systems.
steane1 = CSSCode.from_css_pcm(code_name="steaneCode", hx=H, hz=H, logical_qubits=[(lX, lZ)])
config = {
    "objective_distance": steane1.d,
    "primitives": {
        "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
        "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
        "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
    },
}


program = [
    AllocCode(target_code=steane1, code_varname="steane1", logical_qubits_varnames=["lq_0"]),
    # Initialise both codes in |0_L> (X+ eigenstate)
    InitCode(targets=["steane1"], initial_state=PauliEigenState.Z_plus),
    TransversalH(targets=["steane1"]),
    # Destructive single-code readouts to verify individual logical states
    ReadoutCode(targets=["steane1"]),
    FreeCode(code_varname="steane1"),
]

compiler = QECCompiler(config=config)
compiled_program = compiler.compile(program)

In [ ]:
for i in compiled_program.to_stim_program([]).splitlines():
    print(i)

RZ 6 0 1 3 4 2 5
RZ 8 7 9 11 10 12
REPEAT 3 {
    H 8 7 12
    TICK
    CX 7 2 12 0 8 6
    TICK
    CX 12 2 7 1 8 3
    TICK
    CX 8 2 12 1 7 6
    TICK
    CX 7 5 12 3 8 4
    TICK
    H 8 7 12
    TICK
    CX 1 11 6 10 3 9
    TICK
    CX 1 9 2 11 3 10
    TICK
    CX 6 11 2 9 4 10
    TICK
    CX 5 11 0 9 2 10
    TICK
    MRZ 8 7 9 11 10 12
}
H 6 0 1 3 4 2 5
RZ 8 7 9 11 10 12
REPEAT 3 {
    H 8 7 12
    TICK
    CX 7 2 12 0 8 6
    TICK
    CX 12 2 7 1 8 3
    TICK
    CX 8 2 12 1 7 6
    TICK
    CX 7 5 12 3 8 4
    TICK
    H 8 7 12
    TICK
    CX 1 11 6 10 3 9
    TICK
    CX 1 9 2 11 3 10
    TICK
    CX 6 11 2 9 4 10
    TICK
    CX 5 11 0 9 2 10
    TICK
    MRZ 8 7 9 11 10 12
}
MZ 5 3 6 0 4 2 1
DETECTOR rec[-43] rec[-37] # _det_c_x_0_steaneCode_r0_1
DETECTOR rec[-37] rec[-31] # _det_c_x_0_steaneCode_r1_2
DETECTOR rec[-42] rec[-36] # _det_c_x_2_steaneCode_r0_1
DETECTOR rec[-36] rec[-30] # _det_c_x_2_steaneCode_r1_2
DETECTOR rec[-41] rec[-35] # _det_c_z_1_steaneCode_r0_1
DE